# Transaction Category Model Evaluation
This notebook evaluates the category classifier using a train/test split from labeled transactions in the DB.

In [ ]:
import os
import django

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'budget_project.settings')
# Jupyter runs an event loop; allow Django ORM access in this notebook context.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')
django.setup()

print('Django initialized')

Django initialized


In [2]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

from transactions.category_classifier import TransactionCategoryClassifier

Matplotlib is building the font cache; this may take a moment.
Fontconfig warning: ignoring C.UTF-8: not a valid language tag


In [3]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
THRESHOLDS = [0.30, 0.45, 0.60, 0.75]

clf = TransactionCategoryClassifier()
texts, labels = clf.build_dataset_from_db()
label_counts = Counter(labels)

print(f'Rows: {len(texts)}')
print(f'Categories: {len(label_counts)}')
print('Top categories:', label_counts.most_common(10))

SynchronousOnlyOperation: You cannot call this from an async context - use a thread or sync_to_async.

In [ ]:
if len(texts) < clf.min_samples:
    raise ValueError(f'Need at least {clf.min_samples} labeled rows, found {len(texts)}')
if len(label_counts) < clf.min_classes:
    raise ValueError('Need at least 2 categories for evaluation')

stratify = labels if min(label_counts.values()) >= 2 else None
x_train, x_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify,
)

model = clf.create_pipeline()
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)
max_conf = y_proba.max(axis=1)

print(f'Train rows: {len(x_train)} | Test rows: {len(x_test)}')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Macro F1: {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'Weighted F1: {f1_score(y_test, y_pred, average="weighted"):.4f}')

In [ ]:
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    ax=ax,
    xticks_rotation=45,
    colorbar=False,
    cmap='Blues',
)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
results = []
for threshold in THRESHOLDS:
    idx = np.where(max_conf >= threshold)[0]
    coverage = len(idx) / len(y_test)

    if len(idx) == 0:
        results.append((threshold, coverage, 0, np.nan, np.nan, np.nan))
        continue

    yt = [y_test[i] for i in idx]
    yp = [y_pred[i] for i in idx]
    p, r, f1, _ = precision_recall_fscore_support(
        yt, yp, average='macro', zero_division=0
    )
    results.append((threshold, coverage, len(idx), p, r, f1))

for threshold, coverage, accepted, p, r, f1 in results:
    print(
        f't={threshold:.2f} | coverage={coverage:.3f} | accepted={accepted} | ' \
        f'precision_macro={p:.3f} recall_macro={r:.3f} f1_macro={f1:.3f}'
    )

In [ ]:
thresholds = [r[0] for r in results]
coverage = [r[1] for r in results]
f1s = [r[5] for r in results]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(thresholds, coverage, marker='o', label='Coverage')
ax1.set_xlabel('Confidence Threshold')
ax1.set_ylabel('Coverage')
ax1.set_ylim(0, 1.05)

ax2 = ax1.twinx()
ax2.plot(thresholds, f1s, marker='s', color='tab:orange', label='Macro F1 (accepted)')
ax2.set_ylabel('Macro F1')
ax2.set_ylim(0, 1.05)

ax1.set_title('Threshold Tradeoff: Coverage vs Macro F1')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()